# Notebook 4 — Deduplication and Document Versioning With Ready-Made Tools
## Advanced RAG | Section 4

**Author:** Suresh D R | AI Product Developer & Technology Mentor

---

**The core question:** a PDF is already in the vector database and the **same document is uploaded again**. What happens, how do we stop it, and how do we clean up?

We use the files in **`technova_data/dedup/`** — the kind of mess every company has: an old and a new policy, an exact copy, a re-exported copy, a Word copy, a draft with one word changed, and other documents that share paragraphs.

| Job | Ready-Made Tool |
|---|---|
| PDF / Word → text | `pymupdf4llm`, `MarkItDown` |
| Chunking | LangChain `MarkdownHeaderTextSplitter` |
| Vector database | LangChain `Chroma` |
| **Record manager** — remembers what is stored, skips duplicates, replaces old versions | LangChain **`index()`** + **`SQLRecordManager`** |
| Near-duplicate detection | **`datasketch`** MinHash + LSH |
| Remove repeats at search time | Chroma's built-in **`max_marginal_relevance_search`** (MMR) |

| Step | What We Do | Doc Section |
|---|---|---|
| 1 | Look at the data folder | — |
| 2 | Load files into chunks | — |
| 3 | **No protection** — upload the same PDF twice and see the damage | Core scenario, Part A |
| 4 | **Gatekeeper** — file hash and content hash | 4.1, 4.2 |
| 5 | **Safety net** — LangChain `index()` skips what is already stored | Core scenario, Part B |
| 6 | Near-duplicates — MinHash | 4.3 |
| 7 | Same meaning, different words — embeddings | 4.4 |
| 8 | Same paragraph in many documents — chunk hashes | 4.5 |
| 9 | **The full upload pipeline** + versioning 2024 → 2025 | 4.6, 4.7 |
| 10 | Duplicates already stored — clean up | Core scenario, Part D |
| 11 | MMR — remove repeats at search time | 4.8 |

> Cells that call an AI model are marked **🤖** in their first line (here: only the embedding model). Every other cell is plain Python — hashing, MinHash, chunking — and its output is already shown below.

**Before you start:** `technova_data/` must be next to this notebook (upload/unzip `technova_data.zip` — the next cell does it for you).

---
### Before You Start: Setup (2 minutes)

**1. The data:** you need `technova_data.zip`. The **Get the Data** cell below asks you to upload it (Google Colab) and extracts only the folder this notebook uses.

**2. Your OpenAI API key** — this notebook needs only this one secret:

| Step | What to Do |
|---|---|
| 1 | Go to **platform.openai.com** → sign in → **API keys** → **Create new secret key** |
| 2 | Copy the key (it starts with `sk-`). It is shown **only once** — save it somewhere safe |
| 3 | Make sure your OpenAI account has **billing / credits** enabled (this notebook costs only a few cents) |

**3. How to give the key to the notebook** — choose one:

| Option | How |
|---|---|
| **A. Type it when asked** (simplest) | Run the Setup cell → a hidden box appears → paste the key → press Enter |
| **B. Colab Secrets** (no retyping) | Click the 🔑 **Secrets** icon in Colab's left sidebar → **Add new secret** → name `OPENAI_API_KEY`, value = your key → turn on **Notebook access**. The Setup cell reads it automatically |

⚠️ **Never type your key directly into a code cell** — anyone who gets the notebook file could use it.

In [ ]:
%pip install -q openai langchain langchain-openai langchain-classic langchain-chroma langchain-text-splitters pymupdf4llm "markitdown[docx]" datasketch sqlalchemy pandas

---
## Get the Data (Google Colab or local)

This cell makes sure this notebook's data folder, **`technova_data/dedup/`**, is available:

| Situation | What Happens |
|---|---|
| Folder already there | Nothing to do |
| `technova_data.zip` is next to the notebook | Extracts **only** `technova_data/dedup/` from the zip |
| Running on **Google Colab** with no zip yet | Opens an **upload box** — choose `technova_data.zip` from your computer |

In [ ]:
# 🤖 uses the embedding model (text-embedding-3-small)
import zipfile
from pathlib import Path

NEEDED = "technova_data/dedup/"                       # the only folder this notebook uses

if Path(NEEDED).exists() and any(Path(NEEDED).iterdir()):
    print("Data folder already available.")
else:
    zips = sorted(Path(".").glob("technova_data*.zip"))
    if zips:
        zip_path = zips[0]
    else:
        try:
            from google.colab import files               # only exists on Google Colab
        except ImportError:
            raise FileNotFoundError("Put technova_data.zip next to this notebook, then run this cell again.")
        print("Choose technova_data.zip from your computer:")
        uploaded = files.upload()
        zip_path = Path(next(name for name in uploaded if name.endswith(".zip")))
    with zipfile.ZipFile(zip_path) as z:
        members = [m for m in z.namelist() if m.startswith(NEEDED)]
        if not members:
            raise ValueError(f"{zip_path.name} does not contain {NEEDED}")
        z.extractall(".", members=members)
    print(f"Extracted {len(members)} files from {zip_path.name} -> {NEEDED}")

for p in sorted(Path(NEEDED).iterdir()):
    print(f"  {p.name:<40} {p.stat().st_size / 1024:>8.1f} KB")

In [ ]:
import os, re, getpass, hashlib, shutil
from pathlib import Path
import numpy as np
import pandas as pd
from langchain_openai import OpenAIEmbeddings
from langchain_chroma import Chroma
from langchain_core.documents import Document

pd.set_option("display.max_colwidth", 100)
try:                                                    # Google Colab Secrets (optional)
    from google.colab import userdata
    os.environ.setdefault("OPENAI_API_KEY", userdata.get("OPENAI_API_KEY"))
except Exception:
    pass
if not os.environ.get("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass.getpass("Paste your OpenAI API key: ")
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

DATA = Path("technova_data/dedup")
assert DATA.exists(), "Data not found - run Notebook 0 first, or unzip technova_data.zip next to this notebook."
OUT = Path("nb4_output"); shutil.rmtree(OUT, ignore_errors=True); OUT.mkdir()

def show(title, text=""):
    print(f"\n=== {title} ===")
    if text:
        print(text)
print("Ready. Input folder:", DATA)

Ready. Input folder: technova_data/dedup


---
## Step 1: Look at the Data Folder

| File | What It Really Is | What Should Happen |
|---|---|---|
| `HR_Policy_2024.pdf` | Old policy — sick leave **10 days** | Ingest (version 2024) |
| `HR_Policy_2025.pdf` | New policy — sick leave **12 days** | Replace the 2024 version |
| `HR_Policy_2025 (1).pdf` | Byte-for-byte copy | Skip — exact copy |
| `HR_Policy_2025_reexport.pdf` | Same words, saved again (different bytes) | Skip — same content |
| `HR_Policy_2025.docx` | Same words, Word format | Skip — same content |
| `HR_Policy_2025_draft.pdf` | One word changed ("two" → "three" WFH days) | Flag — near-duplicate |
| `Employee_FAQ.pdf` | Copies the leave section + an FAQ in different words | Ingest |
| `Onboarding_Guide.pdf` | New document; shares the confidentiality paragraph | Ingest |

In [ ]:
FILES = sorted(DATA.iterdir())
pd.DataFrame([{"file": p.name, "size_bytes": p.stat().st_size} for p in FILES])

,file,size_bytes
0,Employee_FAQ.pdf,2745
1,HR_Policy_2024.pdf,2882
2,HR_Policy_2025 (1).pdf,2886
3,HR_Policy_2025.docx,37352
4,HR_Policy_2025.pdf,2886
5,HR_Policy_2025_draft.pdf,2887
6,HR_Policy_2025_reexport.pdf,2896
7,Onboarding_Guide.pdf,1960


---
## Step 2: Load Files Into Chunks

| Step | Tool |
|---|---|
| PDF → Markdown (headers and footers removed) | `pymupdf4llm` |
| Word → Markdown | `MarkItDown` |
| Markdown → one chunk per section, label on top | `MarkdownHeaderTextSplitter` |

**Important for deduplication:** each chunk's metadata holds only `doc_id` and `section` — **not** the file name. The same paragraph in `HR_Policy_2024.pdf` and `HR_Policy_2025.pdf` must look identical, otherwise it would never be recognised as "unchanged".

In [ ]:
import pymupdf4llm
from markitdown import MarkItDown
from langchain_text_splitters import MarkdownHeaderTextSplitter

mid = MarkItDown()
splitter = MarkdownHeaderTextSplitter(headers_to_split_on=[("#", "title"), ("##", "section")])

def to_markdown(path):
    path = Path(path)
    if path.suffix == ".docx":
        md = mid.convert(str(path)).text_content.strip()
        md = re.sub(r"^# ", "## ", md, flags=re.M)                     # Word headings -> sections
        first, rest = md.split("\n", 1)
        return f"# {first}\n{rest}"                                      # Word title -> document title
    return pymupdf4llm.to_markdown(str(path), header=False, footer=False, use_ocr=False)   # digital PDFs: no OCR needed

def load_docs(path, doc_id):
    docs = []
    for d in splitter.split_text(to_markdown(path)):
        title = re.sub(r"[*_]", "", d.metadata.get("title", "")).strip()
        section = re.sub(r"[*_]", "", d.metadata.get("section", "")).strip()
        if d.page_content.strip():
            docs.append(Document(page_content=f"Document: {title} | Section: {section}\n{d.page_content.strip()}",
                                 metadata={"doc_id": doc_id, "section": section}))
    return docs

def normalise(text):
    """Remove Markdown symbols, case and spacing so only the words are compared."""
    return " ".join(re.sub(r"[#*_|`>~]|-{3,}", " ", text).lower().split())

hr25 = load_docs(DATA / "HR_Policy_2025.pdf", "hr-policy")
print(f"HR_Policy_2025.pdf -> {len(hr25)} chunks")
pd.DataFrame([{"section": d.metadata["section"], "chunk": d.page_content.split("\n", 1)[1][:90]} for d in hr25])

HR_Policy_2025.pdf -> 5 chunks


,section,chunk
0,1. About This Policy,This policy applies to all full-time employees of TechNova Solutions Pvt Ltd. It is effect
1,2. Working Hours,"Standard working hours are 9:30 AM to 6:30 PM, Monday to Friday. \nEmployees may work from"
2,3. Leave Policy,Employees receive 12 days of paid sick leave per year. A medical certificate is required f
3,4. Travel and Expenses,"Expense claims above Rs 50,000 need approval from the department head. Claims must be subm"
4,5. Confidentiality,This document is the property of TechNova Solutions Pvt Ltd and is confidential. Do not sh


---
## Step 3 — No Protection: Upload the Same PDF Twice

Basic RAG code calls `add_documents()` on every upload. It creates **new random IDs** every time, and a vector database only checks IDs — never content. So a copy looks like new data.

In [ ]:
# 🤖 uses the embedding model (text-embedding-3-small)
naive = Chroma(collection_name="no_protection", embedding_function=embeddings)
log = []
for day, name in [("Monday", "HR_Policy_2025.pdf"), ("Wednesday", "HR_Policy_2025 (1).pdf")]:
    docs = load_docs(DATA / name, "hr-policy")
    for d in docs:
        d.metadata["file"] = name
    naive.add_documents(docs)
    log.append({"day": day, "upload": name, "chunks added": len(docs), "vector DB size": naive._collection.count()})
display(pd.DataFrame(log))

hits = naive.similarity_search("What is the sick leave policy?", k=4)
top = pd.DataFrame({"rank": range(1, 5), "file": [d.metadata["file"] for d in hits],
                    "chunk": [d.page_content.split("\n", 1)[1][:80] for d in hits]})
show('Search "What is the sick leave policy?" - top 4')
display(top)
print(f"Different chunks in the top 4: {top['chunk'].nunique()} of 4")

**The damage:** every chunk is stored twice, the top 4 holds only 2 different facts, embedding cost doubles, and the same policy is cited under two file names.

---
## Step 4 — Gatekeeper: File Hash and Content Hash (4.1, 4.2)

Check **before** embedding anything.

| Fingerprint | Of What | Catches | Misses |
|---|---|---|---|
| **File hash** | The raw bytes | Exact copies | Re-exports, Word vs PDF |
| **Content hash** | The normalised words | Re-exports, Word vs PDF, renamed files | Any real change |

In [ ]:
def file_hash(path):
    return hashlib.sha256(Path(path).read_bytes()).hexdigest()

def content_hash(path):
    return hashlib.sha256(normalise(to_markdown(path)).encode()).hexdigest()

fp = pd.DataFrame([{"file": p.name, "file_hash": file_hash(p)[:12], "content_hash": content_hash(p)[:12]} for p in FILES])
base = fp[fp.file == "HR_Policy_2025.pdf"].iloc[0]
fp["same file as 2025?"] = np.where(fp.file_hash == base.file_hash, "YES", "")
fp["same content as 2025?"] = np.where(fp.content_hash == base.content_hash, "YES", "")
fp

,file,file_hash,content_hash,same file as 2025?,same content as 2025?
0,Employee_FAQ.pdf,766b3388a6a7,ac7a4b61d1c2,,
1,HR_Policy_2024.pdf,ac3e545ee0ca,0324e48a0765,,
2,HR_Policy_2025 (1).pdf,6f163484d1de,7000ef5aefb1,YES,YES
3,HR_Policy_2025.docx,01f9aaf51f7a,7000ef5aefb1,,YES
4,HR_Policy_2025.pdf,6f163484d1de,7000ef5aefb1,YES,YES
5,HR_Policy_2025_draft.pdf,81601e7fd5fc,d250caec1195,,
6,HR_Policy_2025_reexport.pdf,4a3256e79eed,7000ef5aefb1,,YES
7,Onboarding_Guide.pdf,43b553aa277b,f3049cbe3dc4,,


---
## Step 5 — Safety Net: LangChain `index()` + `SQLRecordManager`

The gatekeeper can be skipped by mistake (a bug, a script, a re-run). The **storage step itself** must refuse copies.

LangChain's **indexing API** does exactly that:

| Part | What It Does |
|---|---|
| `SQLRecordManager` | A small SQL table that remembers a **hash of every chunk** already stored |
| `index(docs, record_manager, vector_store, ...)` | Hashes each chunk → **skips** chunks already stored, **adds** new ones |
| `cleanup="incremental"` + `source_id_key="doc_id"` | When a document is re-indexed, chunks of that `doc_id` that no longer exist are **deleted** (used in Step 9) |

It returns counts: `num_added`, `num_skipped`, `num_updated`, `num_deleted`.

In [ ]:
# 🤖 uses the embedding model (text-embedding-3-small)
from langchain_classic.indexes import SQLRecordManager
from langchain_core.indexing import index

def new_store(name):
    rm = SQLRecordManager(f"technova/{name}", db_url=f"sqlite:///{OUT / 'record_manager.db'}")
    rm.create_schema()
    return rm, Chroma(collection_name=name, embedding_function=embeddings)

rm_demo, safe = new_store("safety_net")
rows = []
for day, name in [("Monday", "HR_Policy_2025.pdf"), ("Wednesday", "HR_Policy_2025 (1).pdf")]:
    r = index(load_docs(DATA / name, "hr-policy"), rm_demo, safe, cleanup="incremental", source_id_key="doc_id", key_encoder="sha256")
    rows.append({"day": day, "upload": name, **r, "vector DB size": safe._collection.count()})
display(pd.DataFrame(rows))
print("Same file again -> every chunk skipped, nothing embedded, no copies.")

---
## Step 6 — Near-Duplicates With MinHash (4.3)

The draft changes **one word**, so both hashes change. MinHash measures **how much** two documents overlap:

| Step | What Happens |
|---|---|
| 1 | Cut the text into overlapping 5-word groups ("shingles") |
| 2 | MinHash compresses each document into 128 numbers |
| 3 | Comparing the numbers estimates the overlap (Jaccard): 1.0 = identical, 0 = nothing shared |
| 4 | LSH finds likely matches without comparing against every document |
| 5 | Overlap ≥ 0.8 → near-duplicate |

In [ ]:
from datasketch import MinHash, MinHashLSH

def minhash(path, k=5):
    words = normalise(to_markdown(path)).split()
    m = MinHash(num_perm=128)
    for i in range(len(words) - k + 1):
        m.update(" ".join(words[i:i + k]).encode())
    return m

sig = {p.name: minhash(p) for p in FILES if p.suffix == ".pdf"}
pairs = [("HR_Policy_2025.pdf", "HR_Policy_2025_draft.pdf"), ("HR_Policy_2025.pdf", "HR_Policy_2024.pdf"),
         ("HR_Policy_2025.pdf", "Employee_FAQ.pdf"), ("HR_Policy_2025.pdf", "Onboarding_Guide.pdf")]
display(pd.DataFrame([{"document A": a, "document B": b, "overlap (Jaccard)": round(sig[a].jaccard(sig[b]), 3),
                       "near-duplicate?": "YES" if sig[a].jaccard(sig[b]) >= 0.8 else "no"} for a, b in pairs]))

lsh = MinHashLSH(threshold=0.8, num_perm=128)
for name in ["HR_Policy_2025.pdf", "Employee_FAQ.pdf", "Onboarding_Guide.pdf"]:
    lsh.insert(name, sig[name])
print("LSH: documents similar to the draft ->", lsh.query(sig["HR_Policy_2025_draft.pdf"]))

,document A,document B,overlap (Jaccard),near-duplicate?
0,HR_Policy_2025.pdf,HR_Policy_2025_draft.pdf,0.930,YES
1,HR_Policy_2025.pdf,HR_Policy_2024.pdf,0.805,YES
2,HR_Policy_2025.pdf,Employee_FAQ.pdf,0.406,no
3,HR_Policy_2025.pdf,Onboarding_Guide.pdf,0.086,no


LSH: documents similar to the draft -> ['HR_Policy_2025.pdf']


---
## Step 7 — Same Meaning, Different Words (4.4)

Embeddings find chunks that **mean** the same thing. But look at the "10 days vs 12 days" row: a changed number barely moves the score.
That is why semantic duplicates are **tagged, not deleted** — and handled at search time with MMR (Step 11).

In [ ]:
# 🤖 uses the embedding model (text-embedding-3-small)
sentences = {"Policy":          "Employees receive 12 days of paid sick leave per year.",
             "FAQ (reworded)":  "You get twelve paid sick days every year.",
             "Policy, 10 days": "Employees receive 10 days of paid sick leave per year.",
             "Unrelated":       "Expense claims above Rs 50,000 need approval from the department head."}
vecs = dict(zip(sentences, map(np.array, embeddings.embed_documents(list(sentences.values())))))
cos = lambda a, b: float(a @ b / (np.linalg.norm(a) * np.linalg.norm(b)))
pd.DataFrame([{"compared with 'Policy'": k, "sentence": s, "cosine similarity": round(cos(vecs["Policy"], vecs[k]), 3)}
              for k, s in sentences.items() if k != "Policy"])

---
## Step 8 — Same Paragraph in Many Documents (4.5)

Hash every chunk's text (without the label) and count how many documents contain it.

| Found In | Action |
|---|---|
| 1 document | Store |
| 2 documents | Store once |
| 3+ documents | **Boilerplate** (like a confidentiality notice) — never answers a question → drop |

In [ ]:
where = {}
for name, doc_id in [("HR_Policy_2025.pdf", "hr-policy"), ("Employee_FAQ.pdf", "employee-faq"),
                     ("Onboarding_Guide.pdf", "onboarding-guide")]:
    for d in load_docs(DATA / name, doc_id):
        body = d.page_content.split("\n", 1)[1]
        where.setdefault(hashlib.sha256(normalise(body).encode()).hexdigest(), {"chunk": body[:70], "files": []})["files"].append(name)

pd.DataFrame([{"chunk": v["chunk"].replace("\n", " "), "found in": ", ".join(v["files"]),
               "action": "DROP - boilerplate" if len(v["files"]) >= 3 else "store once" if len(v["files"]) == 2 else "store"}
              for v in where.values()]).sort_values("action")

,chunk,found in,action
4,This document is the property of TechNova Solutions Pvt Ltd and is con,"HR_Policy_2025.pdf, Employee_FAQ.pdf, Onboarding_Guide.pdf",DROP - boilerplate
0,This policy applies to all full-time employees of TechNova Solutions P,HR_Policy_2025.pdf,store
1,"Standard working hours are 9:30 AM to 6:30 PM, Monday to Friday. Emp",HR_Policy_2025.pdf,store
3,"Expense claims above Rs 50,000 need approval from the department head.",HR_Policy_2025.pdf,store
5,How many sick days do I get? You get twelve paid sick days every year.,Employee_FAQ.pdf,store
6,On day one you receive your laptop and access card. Your manager sets,Onboarding_Guide.pdf,store
2,Employees receive 12 days of paid sick leave per year. A medical certi,"HR_Policy_2025.pdf, Employee_FAQ.pdf",store once


---
## Step 9 — The Full Upload Pipeline + Versioning (4.6, 4.7)

Every upload goes through these checks, in order:

| # | Check | If Yes | Tool |
|---|---|---|---|
| 1 | File hash seen before? | **SKIP** — exact copy | hashlib |
| 2 | Content hash seen before? | **SKIP** — same content | hashlib |
| 3 | Unknown `doc_id` but ≥ 0.8 overlap with a stored document? | **FLAG** — probably a new version, ask a human | MinHash |
| 4 | Otherwise | **INDEX** — `index()` adds new chunks, skips unchanged ones, deletes chunks of this `doc_id` that disappeared | LangChain `index()` |

**Versioning comes for free:** when `HR_Policy_2025.pdf` is indexed with the same `doc_id` as 2024, `index()` keeps the unchanged sections, adds the changed ones and deletes the old ones. The database then holds **one** answer — 12 days.

In [ ]:
# 🤖 uses the embedding model (text-embedding-3-small)
rm, kb = new_store("technova_managed")
seen_files, seen_content, signatures = {}, {}, {}

def upload(path, doc_id):
    path = Path(path)
    fh = file_hash(path)
    if fh in seen_files:
        return {"status": "SKIPPED - exact copy", "detail": f"same file as {seen_files[fh]}"}
    ch = content_hash(path)
    if ch in seen_content:
        return {"status": "SKIPPED - same content", "detail": f"same text as {seen_content[ch]}"}
    sig_ = minhash(path)
    if doc_id not in signatures:                                   # a document we have never seen
        best = max(((sig_.jaccard(s), d) for d, s in signatures.items()), default=(0, None))
        if best[0] >= 0.8:
            return {"status": "FLAGGED - near-duplicate", "detail": f"{best[0]:.2f} overlap with '{best[1]}' - new version?"}
    r = index(load_docs(path, doc_id), rm, kb, cleanup="incremental", source_id_key="doc_id", key_encoder="sha256")
    seen_files[fh], seen_content[ch], signatures[doc_id] = path.name, path.name, sig_
    return {"status": "INDEXED", "detail": f"added {r['num_added']}, skipped (unchanged) {r['num_skipped']}, "
                                           f"deleted (old) {r['num_deleted']}"}

UPLOADS = [("HR_Policy_2024.pdf", "hr-policy"), ("HR_Policy_2025.pdf", "hr-policy"),
           ("HR_Policy_2025 (1).pdf", "hr-policy"), ("HR_Policy_2025_reexport.pdf", "hr-policy"),
           ("HR_Policy_2025.docx", "hr-policy"), ("HR_Policy_2025_draft.pdf", "hr-policy-draft"),
           ("Employee_FAQ.pdf", "employee-faq"), ("Onboarding_Guide.pdf", "onboarding-guide")]
results = [{"#": i, "file": f, "doc_id": d, **upload(DATA / f, d)} for i, (f, d) in enumerate(UPLOADS, 1)]
display(pd.DataFrame(results))
print("Vectors stored:", kb._collection.count())

In [ ]:
# 🤖 uses the embedding model (text-embedding-3-small)
hit = kb.similarity_search("How many days of paid sick leave do employees get?", k=1, filter={"doc_id": "hr-policy"})[0]
show("Answer chunk after the version update (only the 2025 text remains)", hit.page_content)
print("\n'10 days' still stored anywhere?",
      any("10 days of paid sick leave" in t for t in kb._collection.get()["documents"]))

**Hard delete vs history:** `index()` **deletes** the old 2024 chunks — simple, and no stale answers.
If users must ask "what was the policy in 2024?", index each version into an **archive** collection as well (or use soft delete with `is_current` metadata, as explained in Section 4.6).

---
## Step 10 — Duplicates Already Stored: Clean Up (Part D)

The `no_protection` store from Step 3 already holds copies. One-time cleanup:

| Step | What Happens |
|---|---|
| 1 | Read every chunk from the vector database |
| 2 | Hash each chunk's normalised text |
| 3 | Group identical hashes, keep one per group |
| 4 | Delete the rest |
| 5 | From now on, upload only through the pipeline in Step 9 |

In [ ]:
# 🤖 uses the embedding model (text-embedding-3-small)
from collections import defaultdict

before = naive._collection.count()
data = naive._collection.get(include=["documents", "metadatas"])
groups = defaultdict(list)
for cid, text, meta in zip(data["ids"], data["documents"], data["metadatas"]):
    groups[hashlib.sha256(normalise(text).encode()).hexdigest()].append((meta["file"], cid))
to_delete = [cid for members in groups.values() for _, cid in sorted(members)[1:]]   # keep the official file name
if to_delete:
    naive.delete(ids=to_delete)
pd.DataFrame({"": ["Before cleanup", "Duplicate groups", "Deleted copies", "After cleanup"],
              "count": [before, sum(len(m) > 1 for m in groups.values()), len(to_delete), naive._collection.count()]})

---
## Step 11 — MMR: Remove Repeats at Search Time (4.8)

Semantic duplicates remain (the FAQ repeats the policy in different words). **MMR** picks chunks that are relevant **and** different from those already picked.
Chroma has it built in: `max_marginal_relevance_search(query, k, fetch_k, lambda_mult)`.

| `lambda_mult` | Behaviour |
|---|---|
| 1.0 | Pure relevance (same as normal search) |
| 0.5 | Balanced — a common default |
| 0.0 | Pure diversity |

In [ ]:
# 🤖 uses the embedding model (text-embedding-3-small)
q = "What is the sick leave policy?"
plain = kb.similarity_search(q, k=4)
diverse = kb.max_marginal_relevance_search(q, k=4, fetch_k=10, lambda_mult=0.5)
label = lambda d: f"[{d.metadata['doc_id']}] " + d.page_content.split("\n", 1)[1][:80].replace("\n", " ")
pd.DataFrame({"rank": range(1, 5), "before MMR (similarity only)": [label(d) for d in plain],
              "after MMR (lambda 0.5)": [label(d) for d in diverse]})

---
## Summary

| Question | Answer | Tool |
|---|---|---|
| Same PDF uploaded again, no protection? | Every chunk stored twice | `add_documents()` — Step 3 |
| How to stop it before embedding? | File hash → content hash | hashlib — Step 4 |
| What if that check is skipped? | The record manager skips chunks already stored | `index()` + `SQLRecordManager` — Step 5 |
| One word changed? | Near-duplicate → flag | `datasketch` MinHash — Step 6 |
| Same meaning, different words? | Tag, don't delete; MMR at search time | embeddings, MMR — Steps 7, 11 |
| Same paragraph in many documents? | Store once; 3+ documents = boilerplate | chunk hashes — Step 8 |
| New edition of a document? | Same `doc_id` → unchanged skipped, new added, old deleted | `index(cleanup="incremental")` — Step 9 |
| Duplicates already stored? | Hash, group, keep one, delete the rest | Step 10 |